In [1]:
import pandas as pd
import numpy as np 
import math
import yaml
import logging
from sklearn.preprocessing import OneHotEncoder
from typing import Tuple, Any, Dict, List
from src.utils import load_config, load_data, serialize_data, deserialize_data, get_project_root

In [2]:
def save_to_config(key: str, value: any, filename: str = "config.yaml"):
    """
    Menyimpan key dan value baru LANGSUNG ke file config.yaml
    tanpa merusak path yang sudah ada.
    """
    path_config = get_project_root() / "config" / filename
    
    with open(path_config, "r") as file:
        raw_config = yaml.safe_load(file) or {}
        
    raw_config[key] = value
    
    with open(path_config, "w") as file:
        yaml.dump(raw_config, file, default_flow_style=False, sort_keys=True)
        
    print(f"Berhasil menyimpan permanen: '{key}' ke {filename}")

In [3]:
config = load_config()

In [4]:
path_X_train, path_y_train = config["path_train_clean"]
path_X_valid, path_y_valid = config["path_valid_clean"]
path_X_test, path_y_test = config["path_test_clean"]

In [5]:
X_train = deserialize_data(path_X_train)
y_train = deserialize_data(path_X_train)

X_valid = deserialize_data(path_X_valid)
y_valid = deserialize_data(path_X_valid)

X_test = deserialize_data(path_X_test)
y_test = deserialize_data(path_X_test)


In [6]:
X_train.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
29762,45.0,37500.0,MORTGAGE,1.0,DEBTCONSOLIDATION,B,5000.0,11.49,0.13,N,16.0
2714,25.0,50000.0,RENT,5.0,PERSONAL,A,12000.0,7.88,0.24,N,2.0
50,24.0,78000.0,RENT,4.0,DEBTCONSOLIDATION,D,30000.0,10.99,0.38,Y,4.0
28458,31.0,78504.0,RENT,2.0,EDUCATION,C,10000.0,11.41,0.13,N,7.0
3674,26.0,14000.0,RENT,2.0,VENTURE,B,4000.0,10.99,0.29,N,3.0


In [7]:
X_train.isnull().sum()

person_age                    0
person_income                 0
person_home_ownership         0
person_emp_length             0
loan_intent                   0
loan_grade                    0
loan_amnt                     0
loan_int_rate                 0
loan_percent_income           0
cb_person_default_on_file     0
cb_person_cred_hist_length    0
dtype: int64

In [8]:
X_train.dtypes

person_age                    float64
person_income                 float64
person_home_ownership          object
person_emp_length             float64
loan_intent                    object
loan_grade                     object
loan_amnt                     float64
loan_int_rate                 float64
loan_percent_income           float64
cb_person_default_on_file      object
cb_person_cred_hist_length    float64
dtype: object

In [9]:
def fit_ohe(X_train: pd.DataFrame, config: dict) -> OneHotEncoder:
    """
    Melakukan fitting OneHotEncoder langsung pada DataFrame X_train.
    """
    print("Memulai fungsi fit_ohe: Fitting OneHotEncoder pada data latih")
    
    if not isinstance(X_train, pd.DataFrame):
        raise TypeError("Fungsi fit_ohe: X_train harus bertipe pandas DataFrame.")
    if not isinstance(config, dict):
        raise TypeError("Fungsi fit_ohe: config harus bertipe dictionary.")
    
    cat_cols = config.get("columns_cat", []) 
    print(f" -> Kolom kategorikal yang akan diproses: {cat_cols}")
    
    ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
    
    ohe.fit(X_train[cat_cols])
    for col, categories in zip(cat_cols, ohe.categories_):
        print(f" -> Kategori dipelajari untuk '{col}': {categories.tolist()}")
        
    path_simpan = config.get("path_ohe_model", "models/ohe_model.pkl")

    print(f" -> Model OHE berhasil diserialisasi ke: {path_simpan}")
    
    print("Fungsi fit_ohe selesai dieksekusi.")
    
    return ohe

In [10]:
def transform_ohe(X: pd.DataFrame, ohe: OneHotEncoder, config: dict) -> pd.DataFrame:
    """
    Melakukan transformasi OHE dan menggabungkannya dengan kolom numerik.
    """
    print("Memulai fungsi transform_ohe: Transformasi matriks OHE...")
    
    if not isinstance(X, pd.DataFrame):
        raise TypeError("Fungsi transform_ohe: Parameter (X) harus bertipe pd.DataFrame.")
    if not isinstance(ohe, OneHotEncoder):
        raise TypeError("Fungsi transform_ohe: Parameter (ohe) harus bertipe OneHotEncoder.")
    if not isinstance(config, dict):
        raise TypeError("Fungsi transform_ohe: Parameter (config) harus bertipe dictionary.")
        
    cat_cols = config.get("columns_cat", [])
    
    print(" -> Melakukan transformasi One-Hot Encoding.")
    ohe_features = ohe.transform(X[cat_cols])
    ohe_feature_names = ohe.get_feature_names_out(cat_cols)
    
    df_ohe = pd.DataFrame(ohe_features, columns=ohe_feature_names, index=X.index)
    
    X_dropped = X.drop(columns=cat_cols)
    
    print(" -> Menggabungkan hasil OHE dengan kolom numerikal.")

    X_final = pd.concat([X_dropped, df_ohe], axis=1)
    
    print(f" -> Transformasi berhasil. Total fitur sekarang menjadi: {X_final.shape[1]} kolom.")
    print("Fungsi transform_ohe selesai dieksekusi.")
    
    return X_final

In [11]:
ohe_model = fit_ohe(X_train, config)

Memulai fungsi fit_ohe: Fitting OneHotEncoder pada data latih
 -> Kolom kategorikal yang akan diproses: ['person_home_ownership', 'loan_intent', 'loan_grade', 'cb_person_default_on_file']
 -> Kategori dipelajari untuk 'person_home_ownership': ['MORTGAGE', 'OTHER', 'OWN', 'RENT']
 -> Kategori dipelajari untuk 'loan_intent': ['DEBTCONSOLIDATION', 'EDUCATION', 'HOMEIMPROVEMENT', 'MEDICAL', 'PERSONAL', 'VENTURE']
 -> Kategori dipelajari untuk 'loan_grade': ['A', 'B', 'C', 'D', 'E', 'F', 'G']
 -> Kategori dipelajari untuk 'cb_person_default_on_file': ['N', 'Y']
 -> Model OHE berhasil diserialisasi ke: models/ohe_model.pkl
Fungsi fit_ohe selesai dieksekusi.


In [12]:
save_to_config("path_ohe", "models/ohe_model.pkl")

Berhasil menyimpan permanen: 'path_ohe' ke config.yaml


In [13]:
config = load_config()

In [14]:
serialize_data(ohe_model, config["path_ohe"])

In [15]:
ohe_model = deserialize_data(config["path_ohe"])

In [16]:
X_train_encoded = transform_ohe(X_train, ohe_model, config)
X_valid_encoded = transform_ohe(X_valid, ohe_model, config)
X_test_encoded = transform_ohe(X_test, ohe_model, config)

Memulai fungsi transform_ohe: Transformasi matriks OHE...
 -> Melakukan transformasi One-Hot Encoding.
 -> Menggabungkan hasil OHE dengan kolom numerikal.
 -> Transformasi berhasil. Total fitur sekarang menjadi: 26 kolom.
Fungsi transform_ohe selesai dieksekusi.
Memulai fungsi transform_ohe: Transformasi matriks OHE...
 -> Melakukan transformasi One-Hot Encoding.
 -> Menggabungkan hasil OHE dengan kolom numerikal.
 -> Transformasi berhasil. Total fitur sekarang menjadi: 26 kolom.
Fungsi transform_ohe selesai dieksekusi.
Memulai fungsi transform_ohe: Transformasi matriks OHE...
 -> Melakukan transformasi One-Hot Encoding.
 -> Menggabungkan hasil OHE dengan kolom numerikal.
 -> Transformasi berhasil. Total fitur sekarang menjadi: 26 kolom.
Fungsi transform_ohe selesai dieksekusi.


In [17]:
X_train_encoded.head()

,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,person_home_ownership_MORTGAGE,person_home_ownership_OTHER,person_home_ownership_OWN,...,loan_intent_VENTURE,loan_grade_A,loan_grade_B,loan_grade_C,loan_grade_D,loan_grade_E,loan_grade_F,loan_grade_G,cb_person_default_on_file_N,cb_person_default_on_file_Y
29762,45.0,37500.0,1.0,5000.0,11.49,0.13,16.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2714,25.0,50000.0,5.0,12000.0,7.88,0.24,2.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
50,24.0,78000.0,4.0,30000.0,10.99,0.38,4.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
28458,31.0,78504.0,2.0,10000.0,11.41,0.13,7.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3674,26.0,14000.0,2.0,4000.0,10.99,0.29,3.0,0.0,0.0,0.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [18]:
save_to_config(
    key="path_train_encoded",
    value=["data/modeling_ready/X_train_encoded.pkl", "data/modeling_ready/X_train_encoded.pkl"]
)

save_to_config(
    key="path_valid_encoded",
    value=["data/modeling_ready/X_valid_encoded.pkl", "data/modeling_ready/X_valid_encoded.pkl"]
)

save_to_config(
    key="path_test_encoded",
    value=["data/modeling_ready/X_test_encoded.pkl", "data/modeling_ready/X_test_encoded.pkl"]
)

Berhasil menyimpan permanen: 'path_train_encoded' ke config.yaml
Berhasil menyimpan permanen: 'path_valid_encoded' ke config.yaml
Berhasil menyimpan permanen: 'path_test_encoded' ke config.yaml


In [19]:
config = load_config()

In [20]:
path_X_train_encoded, path_y_train_encoded = config["path_train_encoded"]
path_X_valid_encoded, path_y_valid_encoded = config["path_valid_encoded"]
path_X_test_encoded, path_y_test_encoded = config["path_test_encoded"]

In [21]:
serialize_data(X_train_encoded, path_X_train_encoded)
serialize_data(y_train, path_y_train_encoded)

serialize_data(X_valid_encoded, path_X_valid_encoded)
serialize_data(y_valid, path_y_valid_encoded)

serialize_data(X_test_encoded, path_X_test_encoded)
serialize_data(y_test, path_y_test_encoded)